# 🎧 Playlist Creator: model training (Google Colab)

This notebook trains every model the app uses. **Nothing needs to be uploaded**: all audio is streamed from links.

| Part | What it trains | Data (linked, not uploaded) | Time |
|---|---|---|---|
| 1 | Loads **YAMNet**, a CNN pretrained by Google on millions of YouTube clips. It is the app's "ears" | Hugging Face | 1 min |
| 2 | **Genre**: a small neural-network head + LightGBM on YAMNet embeddings, ensembled | GTZAN (999 clips, 10 genres) from Hugging Face | ~15 min |
| 3 | **Mood**: LightGBM regressors for danceability, energy, valence, acousticness | `dataset.csv` from GitHub + 30 s Deezer previews | ~1 h |

**Why this architecture (transfer learning):** training a CNN from scratch on 1,000 clips overfits (~65–75% genre accuracy).
YAMNet has already learned what music *sounds like*, so we keep it frozen, turn each song into 1,024 numbers (an *embedding*),
and train only small, fast models on top. That typically reaches ~80–90% on GTZAN and trains in minutes.

**Before you start:** `Runtime → Change runtime type → CPU` is fine (a GPU barely helps here). Run the cells top to bottom.
Progress is saved to your Google Drive, so if Colab disconnects, just re-run from the top and it resumes.

In [ ]:
!pip -q install onnxruntime lightgbm librosa pyarrow

In [ ]:
import io, json, os, tempfile, time, warnings
from concurrent.futures import ThreadPoolExecutor, as_completed

import lightgbm as lgb
import librosa
import matplotlib.pyplot as plt
import numpy as np
import onnxruntime as ort
import pandas as pd
import requests
import soundfile as sf

warnings.filterwarnings("ignore")
SR = 16000          # YAMNet expects 16 kHz mono audio
CLIP_SECONDS = 30   # previews and GTZAN clips are 30 s
SEED = 42

In [ ]:
# Everything this notebook produces (caches + final models) goes to Google Drive.
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/playlist-creator"
os.makedirs(OUT, exist_ok=True)
print("Saving to", OUT)

## Part 1: the "ears" (pretrained YAMNet)

YAMNet takes a waveform and, every 0.48 s, outputs a 1,024-number embedding describing the sound.
For a 30 s song we get ~62 embeddings; we summarize them with their **mean** (the overall sound) and **standard deviation** (how much it changes).

We also add 5 rhythm/loudness numbers from Librosa (tempo, loudness and beat strength), because danceability and energy depend on them.
**Total: 2,053 features per song.** The app will compute exactly the same features later.

In [ ]:
YAMNET_URL = "https://huggingface.co/audiomagic/yamnet-onnx/resolve/main/yamnet.onnx"  # Google's YAMNet, converted to ONNX
if not os.path.exists("yamnet.onnx"):
    with open("yamnet.onnx", "wb") as f:
        f.write(requests.get(YAMNET_URL, timeout=300).content)
yamnet = ort.InferenceSession("yamnet.onnx", providers=["CPUExecutionProvider"])

EXTRA_NAMES = ["tempo", "rms_mean", "rms_std", "onset_mean", "pulse_clarity"]
FEATURE_NAMES = ([f"emb_mean_{i}" for i in range(1024)] + [f"emb_std_{i}" for i in range(1024)] + EXTRA_NAMES)


def extract_features(y):
    # y: mono float32 waveform at 16 kHz, values in [-1, 1]. Returns 2,053 numbers.
    y = np.asarray(y, dtype=np.float32)[: SR * CLIP_SECONDS]
    _, emb, _ = yamnet.run(None, {"waveform": y})
    onset = librosa.onset.onset_strength(y=y, sr=SR)
    tempo, beats = librosa.beat.beat_track(onset_envelope=onset, sr=SR)
    tempo = float(np.atleast_1d(tempo)[0])
    pulse = float(onset[beats].mean() / (onset.mean() + 1e-8)) if len(beats) else 0.0
    rms = librosa.feature.rms(y=y)[0]
    extras = np.array([tempo, rms.mean(), rms.std(), onset.mean(), pulse], dtype=np.float32)
    return np.nan_to_num(np.concatenate([emb.mean(0), emb.std(0), extras]).astype(np.float32))


# Sanity check on 3 s of a test tone
print(len(FEATURE_NAMES), extract_features(np.sin(np.linspace(0, 3000, SR * 3)).astype(np.float32)).shape)

## Part 2: genre (GTZAN)

GTZAN has 999 clips across 10 genres. It is streamed from Hugging Face in 3 parquet files of about 440 MB each, and each file is deleted after use.

**Fair evaluation:** GTZAN has many songs by the same artist. If an artist appears in both training and test data, the model can
"recognize the artist" instead of the genre, which inflates accuracy. So we split **by artist**: no artist appears on both sides.

In [ ]:
GTZAN_FILES = [f"https://huggingface.co/datasets/danilotpnta/GTZAN_genre_classification/resolve/main/gtzan/train-0000{i}-of-00003.parquet"
               for i in range(3)]
GTZAN_CACHE = f"{OUT}/gtzan_features.parquet"


def download(url, path):
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        with open(path, "wb") as f:
            for chunk in r.iter_content(1 << 20):
                f.write(chunk)


def load_gtzan_features():
    if os.path.exists(GTZAN_CACHE):
        print("Using cached features from Drive")
        return pd.read_parquet(GTZAN_CACHE)
    rows, start = [], time.time()
    for url in GTZAN_FILES:
        local = url.rsplit("/", 1)[-1]
        print(f"Downloading {local} ...")
        download(url, local)
        part = pd.read_parquet(local, columns=["id", "audio", "genre", "artist"])
        for r in part.itertuples():
            try:
                y, sr = sf.read(io.BytesIO(r.audio["bytes"]), dtype="float32")
            except Exception as exc:  # GTZAN has a known corrupt clip
                print("skip", r.id, exc)
                continue
            if y.ndim > 1:
                y = y.mean(axis=1)
            y = librosa.resample(y, orig_sr=sr, target_sr=SR)
            artist = r.artist if isinstance(r.artist, str) and r.artist.strip() else f"unknown_{r.id}"
            rows.append({"id": r.id, "genre": r.genre, "artist": artist, **dict(zip(FEATURE_NAMES, extract_features(y)))})
        os.remove(local)
        del part
        print(f"  {len(rows)} clips done ({time.time() - start:.0f}s)")
    df = pd.DataFrame(rows)
    df.to_parquet(GTZAN_CACHE, index=False)
    return df


gtzan = load_gtzan_features()
print(gtzan["genre"].value_counts())
print("artists:", gtzan["artist"].nunique())

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold

X = gtzan[FEATURE_NAMES].to_numpy(np.float32)
GENRES = sorted(gtzan["genre"].unique())
y = np.array([GENRES.index(g) for g in gtzan["genre"]])
groups = gtzan["artist"].to_numpy()


def artist_split(idx, seed):
    # Split the rows `idx` ~80/20 so that no artist is on both sides.
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    a, b = next(sgkf.split(X[idx], y[idx], groups[idx]))
    return idx[a], idx[b]


train_idx, test_idx = artist_split(np.arange(len(y)), SEED)
fit_idx, val_idx = artist_split(train_idx, SEED + 1)  # validation set for early stopping (test stays untouched)
assert not set(groups[train_idx]) & set(groups[test_idx])
print(f"train {len(fit_idx)} | validation {len(val_idx)} | test {len(test_idx)}")

### Model A: LightGBM on the embeddings
Gradient-boosted decision trees. `colsample_bytree=0.3` makes each tree look at a random 30% of the 2,053 features, which helps when there are many features and few songs.

In [ ]:
LGBM_GENRE_PARAMS = dict(n_estimators=2000, learning_rate=0.03, num_leaves=15, colsample_bytree=0.3,
                         subsample=0.8, subsample_freq=1, random_state=SEED, verbose=-1)
genre_lgbm = lgb.LGBMClassifier(**LGBM_GENRE_PARAMS)
genre_lgbm.fit(X[fit_idx], y[fit_idx], eval_set=[(X[val_idx], y[val_idx])],
               callbacks=[lgb.early_stopping(100, verbose=False)])
best_trees = genre_lgbm.best_iteration_
p_lgbm = genre_lgbm.predict_proba(X[test_idx])
print(f"LightGBM: {best_trees} trees, test accuracy {(p_lgbm.argmax(1) == y[test_idx]).mean():.3f}")

### Model B: neural-network head
A small dense network (2,053 → 256 → 10) trained on the same embeddings. Inputs are standardized (mean 0, std 1) first, and dropout randomly switches off neurons during training to fight overfitting.

In [ ]:
import tensorflow as tf

tf.keras.utils.set_random_seed(SEED)
MU, SD = X[train_idx].mean(0), X[train_idx].std(0) + 1e-6
scale = lambda a: (a - MU) / SD


def build_head():
    head = tf.keras.Sequential([
        tf.keras.layers.Input((X.shape[1],)),
        tf.keras.layers.Dropout(0.2),
        tf.keras.layers.Dense(256, activation="relu"),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(len(GENRES), activation="softmax"),
    ])
    head.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return head


head = build_head()
stop = tf.keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=15, restore_best_weights=True)
hist = head.fit(scale(X[fit_idx]), y[fit_idx], validation_data=(scale(X[val_idx]), y[val_idx]),
                epochs=200, batch_size=32, verbose=0, callbacks=[stop])
best_epochs = int(np.argmax(hist.history["val_accuracy"])) + 1
p_head = head.predict(scale(X[test_idx]), verbose=0)
print(f"NN head: best epoch {best_epochs}, test accuracy {(p_head.argmax(1) == y[test_idx]).mean():.3f}")

plt.plot(hist.history["accuracy"], label="train")
plt.plot(hist.history["val_accuracy"], label="validation")
plt.xlabel("epoch"); plt.ylabel("accuracy"); plt.legend(); plt.title("NN head training"); plt.show()

### Ensemble: average both models' probabilities

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, classification_report

p_ens = (p_lgbm + p_head) / 2
genre_results = {name: float((p.argmax(1) == y[test_idx]).mean())
                 for name, p in [("lightgbm", p_lgbm), ("nn_head", p_head), ("ensemble", p_ens)]}
print(genre_results)
print(classification_report(y[test_idx], p_ens.argmax(1), labels=range(len(GENRES)), target_names=GENRES, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y[test_idx], p_ens.argmax(1), labels=range(len(GENRES)), display_labels=GENRES, xticks_rotation=45)
plt.title("Genre ensemble: test set (unseen artists)"); plt.show()

### Final genre models: retrain on all 999 clips
Evaluation is done. The models we ship use all the data, keeping the tree count and epoch count found above.

In [ ]:
final_lgbm = lgb.LGBMClassifier(**{**LGBM_GENRE_PARAMS, "n_estimators": best_trees}).fit(X, y)
MU, SD = X.mean(0), X.std(0) + 1e-6
tf.keras.utils.set_random_seed(SEED)
final_head = build_head()
final_head.fit(scale(X), y, epochs=best_epochs, batch_size=32, verbose=0)

final_lgbm.booster_.save_model(f"{OUT}/genre_lgbm.txt")
final_head.save(f"{OUT}/genre_head.keras")
np.savez(f"{OUT}/genre_scaler.npz", mu=MU, sd=SD)
with open(f"{OUT}/genre_meta.json", "w") as f:
    json.dump({"genres": GENRES, "test_accuracy": genre_results, "feature_names": FEATURE_NAMES,
               "sample_rate": SR, "clip_seconds": CLIP_SECONDS}, f)
print("Saved genre models to", OUT)

## Part 3: mood (danceability, energy, valence, acousticness)

`dataset.csv` (from your GitHub repo) has ~114k songs with Spotify's original mood scores, which are the **labels** the models learn from.
For 45 songs per genre (~5,000) we stream the 30 s Deezer preview, compute the same 2,053 features, and train one LightGBM regressor per mood.

This is the slow part (~1 h). It saves to Drive every 200 songs; if Colab disconnects, re-run and it continues where it stopped.

In [ ]:
DATASET_URL = "https://raw.githubusercontent.com/Edwin-Trejo/Automated-playlist-creator/main/data/dataset.csv"
TARGETS = ["danceability", "energy", "valence", "acousticness"]
PER_GENRE = 45

songs = pd.read_csv(DATASET_URL)
songs = songs.dropna(subset=["track_name", "artists"]).drop_duplicates("track_id")
sample = songs.sample(frac=1, random_state=SEED).groupby("track_genre").head(PER_GENRE).reset_index(drop=True)
print(f"{len(songs)} songs in dataset.csv, sampling {len(sample)} ({PER_GENRE} per genre x {sample['track_genre'].nunique()} genres)")
sample[["track_name", "artists", "track_genre"] + TARGETS].head()

In [ ]:
DEEZER = "https://api.deezer.com/search"


def deezer_search(query):
    for attempt in range(4):
        try:
            data = requests.get(DEEZER, params={"q": query}, timeout=10).json()
        except Exception:
            time.sleep(2 * (attempt + 1))
            continue
        if (data.get("error") or {}).get("code") == 4:  # Deezer rate limit: wait and retry
            time.sleep(2 * (attempt + 1))
            continue
        return data.get("data", [])
    return []


def find_preview(track, artist):
    track, artist = track.replace('"', ""), artist.replace('"', "")
    for item in deezer_search(f'artist:"{artist}" track:"{track}"'):
        if item.get("preview"):
            return item["preview"]
    wanted = artist.lower()  # loose search: only accept results by the same artist
    for item in deezer_search(f"{track} {artist}"):
        found = (item.get("artist") or {}).get("name", "").lower()
        if item.get("preview") and found and (found in wanted or wanted in found):
            return item["preview"]
    return None


def load_preview(url):
    data = requests.get(url, timeout=15).content
    with tempfile.NamedTemporaryFile(suffix=".mp3", delete=False) as f:
        f.write(data)
    try:
        y, _ = librosa.load(f.name, sr=SR, mono=True, duration=CLIP_SECONDS)
    finally:
        os.remove(f.name)
    return y if len(y) and np.abs(y).max() > 0 else None


def process(row):
    try:
        artist = str(row["artists"]).split(";")[0]
        url = find_preview(str(row["track_name"]), artist)
        y = load_preview(url) if url else None
    except Exception:
        y = None
    if y is None:
        return {"track_id": row["track_id"], "ok": False}
    return {"track_id": row["track_id"], "ok": True, "artist": artist, "track_genre": row["track_genre"],
            **{t: float(row[t]) for t in TARGETS}, **dict(zip(FEATURE_NAMES, extract_features(y)))}


r = process(sample.iloc[0].to_dict())
print("test song:", sample.iloc[0]["track_name"], "->", "ok" if r["ok"] else "no preview found")

In [ ]:
MOOD_CACHE, FAILED_CACHE = f"{OUT}/mood_features.parquet", f"{OUT}/mood_failed.csv"


def save_progress(rows, failed):
    if rows:
        new = pd.DataFrame(rows)
        if os.path.exists(MOOD_CACHE):
            new = pd.concat([pd.read_parquet(MOOD_CACHE), new], ignore_index=True)
        new.to_parquet(MOOD_CACHE, index=False)
    if failed:
        pd.DataFrame({"track_id": failed}).to_csv(FAILED_CACHE, mode="a", header=not os.path.exists(FAILED_CACHE), index=False)


done = set(pd.read_parquet(MOOD_CACHE)["track_id"]) if os.path.exists(MOOD_CACHE) else set()
if os.path.exists(FAILED_CACHE):
    done |= set(pd.read_csv(FAILED_CACHE)["track_id"])
todo = [r for r in sample.to_dict("records") if r["track_id"] not in done]
print(f"{len(done)} already processed, {len(todo)} to go")

rows, failed, start = [], [], time.time()
with ThreadPoolExecutor(max_workers=8) as pool:
    futures = [pool.submit(process, r) for r in todo]
    for i, fut in enumerate(as_completed(futures), 1):
        res = fut.result()
        if res["ok"]:
            rows.append(res)
        else:
            failed.append(res["track_id"])
        if i % 200 == 0 or i == len(todo):
            save_progress(rows, failed)
            rows, failed = [], []
            rate = i / (time.time() - start)
            print(f"{i}/{len(todo)} songs | {rate:.1f}/s | ~{(len(todo) - i) / rate / 60:.0f} min left")

mood = pd.read_parquet(MOOD_CACHE)
print(f"Usable songs with previews: {len(mood)}")

### Train the mood regressors
Same idea as the genre split: songs by the same artist stay on the same side, so the test score reflects **new** artists.
We compare each model against a baseline that always predicts the average; a model is only useful if it clearly beats it.

In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import GroupShuffleSplit

Xm = mood[FEATURE_NAMES].to_numpy(np.float32)
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED).split(Xm, groups=mood["artist"]))
LGBM_MOOD_PARAMS = dict(n_estimators=3000, learning_rate=0.02, num_leaves=31, colsample_bytree=0.3,
                        subsample=0.8, subsample_freq=1, min_child_samples=20, random_state=SEED, verbose=-1)

mood_results, mood_trees, preds = {}, {}, {}
for t in TARGETS:
    yt = mood[t].to_numpy()
    inner_tr, inner_val = next(GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
                               .split(Xm[tr], groups=mood["artist"].iloc[tr]))
    m = lgb.LGBMRegressor(**LGBM_MOOD_PARAMS)
    m.fit(Xm[tr][inner_tr], yt[tr][inner_tr], eval_set=[(Xm[tr][inner_val], yt[tr][inner_val])],
          callbacks=[lgb.early_stopping(200, verbose=False)])
    preds[t] = np.clip(m.predict(Xm[te]), 0, 1)
    mood_trees[t] = m.best_iteration_
    mood_results[t] = {"mae": mean_absolute_error(yt[te], preds[t]),
                       "baseline_mae": mean_absolute_error(yt[te], np.full(len(te), yt[tr].mean())),
                       "r2": r2_score(yt[te], preds[t])}

results = pd.DataFrame(mood_results).T.round(3)
results["previous (Librosa-only, local)"] = {"danceability": 0.612, "energy": 0.737, "valence": 0.531, "acousticness": 0.691}
print("r2 = share of variation explained (1.0 = perfect); previous column is R2 from the first model")
results

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, t in zip(axes, TARGETS):
    ax.scatter(mood[t].to_numpy()[te], preds[t], s=4, alpha=0.4)
    ax.plot([0, 1], [0, 1], "r--")
    ax.set_title(f"{t} (R2 {mood_results[t]['r2']:.2f})"); ax.set_xlabel("Spotify value"); ax.set_ylabel("predicted")
plt.tight_layout(); plt.show()

### Final mood models: retrain on all songs and save

In [ ]:
for t in TARGETS:
    final = lgb.LGBMRegressor(**{**LGBM_MOOD_PARAMS, "n_estimators": mood_trees[t]}).fit(Xm, mood[t].to_numpy())
    final.booster_.save_model(f"{OUT}/mood_{t}.txt")
with open(f"{OUT}/mood_meta.json", "w") as f:
    json.dump({"targets": TARGETS, "test_results": mood_results, "songs": len(mood),
               "feature_names": FEATURE_NAMES, "sample_rate": SR, "clip_seconds": CLIP_SECONDS}, f)
print("Saved mood models to", OUT)

## Done: download the models

The cell below zips the final models (not the large feature caches). Unzip it into the repo's `models/` folder;
connecting them to the app is the next step we'll do together.

In [ ]:
import shutil
bundle = "/content/playlist_models"
os.makedirs(bundle, exist_ok=True)
for name in ["genre_lgbm.txt", "genre_head.keras", "genre_scaler.npz", "genre_meta.json", "mood_meta.json"] + [f"mood_{t}.txt" for t in TARGETS]:
    shutil.copy(f"{OUT}/{name}", bundle)
shutil.copy("yamnet.onnx", bundle)
shutil.make_archive("/content/playlist_models", "zip", bundle)
from google.colab import files
files.download("/content/playlist_models.zip")